# Numerical Integration with Non-Uniform Grids

This notebook illustrates some basic usage of molecular grids and defines the integrals we aim to solve in the practical project.

First, we install the `qc-grid` package, if not done already:

In [1]:
! pip install qc-grid


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## Definition of molecules and basis sets

Next, a class for molecular data and two dictionaries defining a minimal (`STO_3G`) and slightly larger (`def2-SVP`) basis set. The molecule of interest will be $H_2$. 

In [1]:
from dataclasses import dataclass
import numpy as np
from typing import List, Dict

BOHR = 0.5291772 # Angstrom
Z = {'H' :1,                                            'He':2, 
     'Li':3, 'Be':4, 'B':5, 'C':6, 'N':7, 'O':8, 'F':9, 'Ne':10}

@dataclass
class Molecule:
    elements: List[str]
    coordinates: np.ndarray 
    @property
    def atomic_numbers(self):
        '''Atomic numbers'''
        Zs = [Z[el] for el in self.elements]
        return Zs
    
mol = Molecule(['H','H'],
               np.array([[0.0,0.0,0.0],[0.0,0.0,0.74]])/BOHR)
print(mol)

STO_3G    = {'H':[{"angular_momentum": 0, 
                  "exponents":    np.array([3.42525091, 0.62391373, 0.1688554 ]), 
                  "coefficients": np.array([0.15432897, 0.53532814, 0.44463454]) }]}

def2_SVP = {'H':[{"angular_momentum": 0, 
                  "exponents":    np.array([13.010701  ,  1.9622572 ,  0.44453796 ]), 
                  "coefficients": np.array([0.019682158,  0.13796524, 0.47831935]) },
                 {"angular_momentum": 0, 
                  "exponents":    np.array([0.12194962]), 
                  "coefficients": np.array([1.]) },
                 {"angular_momentum": 1, 
                  "exponents":    np.array([0.8]), 
                  "coefficients": np.array([1.]) }
                ]}

Molecule(elements=['H', 'H'], coordinates=array([[0.        , 0.        , 0.        ],
       [0.        , 0.        , 1.39839736]]))


Now we define some helper functions for working with basis functions. Specifically, we have functions for normalization and for dealing with *shells* of basis functions. Remember that for angular momenta $l>0$ there are always multiple functions (e.g. $p_x$, $p_y$, and $p_z$ for $l=1$).

We will work with a special form of GTOs, namely cartesian GTOs. These replace the spherical harmonics for the angulat part of the basis function with simple polynomials in the cartesian directions $x$, $y$, and $z$. A primitive cartesian GTO with angular momentum $l=i+j+k$ is then defined as:

$$
\chi_\mu (\mathbf{r}) = G_{ijk} (\mathbf{r}) = \mathcal{N} x^i y^j z^k e^{-\alpha|\mathbf{r}|^2} 
$$

Here, $\mathcal{N}$ is a normalization factor.

The class `CGTO` defines a cartesian, contracted GTO of a given angular momentum and returns the values for *all* possible functions of a given $l$ (e.g. three in the case of $l=1$).

In [2]:
import scipy.special

def gaussian_int(n, alpha):
    r'''int_0^inf x^n exp(-alpha x^2) dx'''
    assert n >= 0
    n1 = (n + 1) * .5
    return scipy.special.gamma(n1) / (2. * alpha**n1)

def gto_norm(l, expnt):
    '''Radial part normalization'''
    assert l >= 0
    # Radial part normalization
    norm = (gaussian_int(l*2+2, 2*expnt)) ** -.5
    # Racah normalization, assuming angular part is normalized to unity
    norm *= ((2*l+1)/(4*np.pi))**.5
    return norm    

def gto_offsets(gtos):
    '''Offsets are the position of the first GTO function for each GTO shell
    inside all GTO basis functions. The last element of the offsets is the
    total number of basis functions.
    '''
    dims = [n_cart(b.angular_momentum) for b in gtos]
    return np.append(0, np.cumsum(dims))

def n_cart(l):
    return (l + 1) * (l + 2) // 2

def iter_cart_xyz(n):
    '''Produces all (ix,iy,iz) subject to 0 <= ix/iy/iz <= n and ix + iy + iz == n'''
    return [(i, j, n-i-j)
            for i in reversed(range(n+1))
            for j in reversed(range(n+1-i))]

# Contracted GTOs
@dataclass
class CGTO:
    angular_momentum: int
    exponents:    np.ndarray
    coefficients: np.ndarray
    coordinates:  np.ndarray
    
    @property
    def norm_coefficients(self):
        '''contraction coefficients with normalization factors'''
        norm = gto_norm(self.angular_momentum, self.exponents)
        return norm * (self.coefficients)
    
    def __call__(self,r):
        r_c = r-self.coordinates
        cart_xyz = iter_cart_xyz(self.angular_momentum)
        
        r_norm = np.linalg.norm(r_c,axis=-1)
        radial = np.zeros_like(r_norm)
        for ic, coef in enumerate(self.norm_coefficients):
            radial += coef*np.exp(-self.exponents[ic]*r_norm**2)
        grids = []
        for ix,iy,iz in cart_xyz:
            angular = r_c[:,0]**ix * r_c[:,1]**iy * r_c[:,2]**iz
            grids.append(angular*radial)
        return grids

Finally, we can combine the molecular and basis function definition to yield a basis set in terms of a list of CGTOs for all atoms in our molecule:

In [7]:
def assign_basis(molecule: Molecule, 
                 basis_set: Dict[str, List[Dict]]) -> List[CGTO]:
    '''Creates a list of CGTOs for a molecule object
    '''
    gtos = []
    for elem, r in zip(molecule.elements, molecule.coordinates):
        basis_list = basis_set[elem]
        for basis_function in basis_list:
            gtos.append(CGTO(basis_function["angular_momentum"],
                             basis_function["exponents"],
                             basis_function["coefficients"],
                             r))
    return gtos

basis_set = assign_basis(mol,def2_SVP)

print(f"Number of basis functions: {len(basis_set)}")
for cgto in basis_set:
    print(cgto)

Number of basis functions: 6
CGTO(angular_momentum=0, exponents=array([13.010701  ,  1.9622572 ,  0.44453796]), coefficients=array([0.01968216, 0.13796524, 0.47831935]), coordinates=array([0., 0., 0.]))
CGTO(angular_momentum=0, exponents=array([0.12194962]), coefficients=array([1.]), coordinates=array([0., 0., 0.]))
CGTO(angular_momentum=1, exponents=array([0.8]), coefficients=array([1.]), coordinates=array([0., 0., 0.]))
CGTO(angular_momentum=0, exponents=array([13.010701  ,  1.9622572 ,  0.44453796]), coefficients=array([0.01968216, 0.13796524, 0.47831935]), coordinates=array([0.        , 0.        , 1.39839736]))
CGTO(angular_momentum=0, exponents=array([0.12194962]), coefficients=array([1.]), coordinates=array([0.        , 0.        , 1.39839736]))
CGTO(angular_momentum=1, exponents=array([0.8]), coefficients=array([1.]), coordinates=array([0.        , 0.        , 1.39839736]))


Passing an array of grid points to a `CGTO` returns the value of the basis function(s) corresponding to this CGTO shell on all grid points:

In [9]:
r = np.array([[0.0,0.0,0.0],[0.0,0.0,0.2],[0.0,0.0,0.4],[0.0,0.0,0.8]]) 
basis_set[2](r)

[array([0., 0., 0., 0.]),
 array([0., 0., 0., 0.]),
 array([0.        , 0.20889841, 0.37955339, 0.51705148])]

## Working with molecular grids

Now we can use the `grid` library to construct the grids for us:

In [10]:
from grid import GaussLegendre, BeckeRTransform, BeckeWeights, MolGrid

In [12]:
# construct a radial grid from a 1D grid on [-1, 1] and then transforming it to [0, infinity)
oned_grid = GaussLegendre(npoints=50)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned_grid)

# construct a molecular grid using the Becke partitioning scheme
mgrid = MolGrid.from_preset(
    atnums=mol.atomic_numbers,
    atcoords=mol.coordinates,
    # the same radial grid is used for all atoms
    rgrid=rgrid,
    # different angular degrees for each atom
    preset="fine",
    aim_weights=BeckeWeights(),
    # store the individual atomic grids (used for interpolation)
    store=True,
)

print(f"Molecular grid size             : {mgrid.size}")
print(f"Molecular grid atomic grid sizes  : {[atgrid.size for atgrid in mgrid.atgrids]}")
print(f"Molecular grid points shape     : {mgrid.points.shape}")
print(f"Molecular grid weights shape    : {mgrid.weights.shape}")
print(f"Molecular grid centers: \n{mgrid.atcoords}")

Molecular grid size             : 6320
Molecular grid atomic grid sizes  : [3160, 3160]
Molecular grid points shape     : (6320, 3)
Molecular grid weights shape    : (6320,)
Molecular grid centers: 
[[0.         0.         0.        ]
 [0.         0.         1.39839736]]


As a simple exercise we can now compute overlap integrals:

$$
S_{\mu \nu} = \int d^3\mathbf{r} \chi_{\mu} (\mathbf{r})  \chi_{\nu} (\mathbf{r}) = \int d^3\mathbf{r} \Omega_{\mu \nu} (\mathbf{r})  
$$

Here, we introduced a shorthand notation for **overlap densities** $\Omega_{\mu \nu} (\mathbf{r})  $ between two basis functions, since these will become important later.

To compute overlap integrals, we compute the values of the corresponding basis functions on our grid, multiply them and sum up (weighted by the quadrature weights corresponding to the grid):

In [13]:
ovlp_matrix = np.zeros((10,10))
offs = gto_offsets(basis_set)

for i1,bf1 in enumerate(basis_set):
    mu = bf1(mgrid.points)
    for i2,bf2 in enumerate(basis_set):
        nu = bf2(mgrid.points)
        for imu, mui in enumerate(mu):
            for inu, nui in enumerate(nu):
                ovlp = np.sum(mui*nui*mgrid.weights)
                ovlp_matrix[offs[i1]+imu,offs[i2]+inu] = ovlp
            
print(np.round(ovlp_matrix,decimals=3)) 

[[ 0.345  0.403  0.     0.     0.     0.197  0.332  0.     0.    -0.302]
 [ 0.403  1.     0.     0.    -0.     0.332  0.888  0.    -0.    -0.15 ]
 [ 0.     0.     1.    -0.    -0.    -0.    -0.     0.457 -0.     0.   ]
 [ 0.     0.    -0.     1.     0.    -0.     0.    -0.     0.457 -0.   ]
 [ 0.    -0.    -0.     0.     1.     0.302  0.15   0.    -0.    -0.258]
 [ 0.197  0.332 -0.    -0.     0.302  0.345  0.403 -0.    -0.    -0.   ]
 [ 0.332  0.888 -0.     0.     0.15   0.403  1.    -0.    -0.     0.   ]
 [ 0.     0.     0.457 -0.     0.    -0.    -0.     1.    -0.    -0.   ]
 [ 0.    -0.    -0.     0.457 -0.    -0.    -0.    -0.     1.     0.   ]
 [-0.302 -0.15   0.    -0.    -0.258 -0.     0.    -0.     0.     1.   ]]


Some things to note here:

1) As you can see, the diagonal terms of the overlap matrix are not all one. This is because the primitive functions are normalized, but not the contracted versions. In the `def2-SVP` basis, the first hydrogen s-orbital is a contraction of three primitive Gaussians. We could normalize it with the computed overlap, but this is actually not essential in our further computations.

2) We use the `gto_offsets` function for bookkeeping. This is necessary because some basis shells (the p-orbitals in this case) return multiple individual basis functions (e.g. $p_x$, $p_y$, $p_z$).

## The two-electron integrals

With this, we have all the basic infrastructure to compute the two electron integrals as defined in the practical project description:

$$
(\mu \nu | \sigma \lambda) = \iint d^3\mathbf{r} d^3\mathbf{r'} \Omega_{\mu \nu} (\mathbf{r})  \frac{1}{||\mathbf{r}-\mathbf{r'}||} \Omega_{\sigma \lambda} (\mathbf{r'})
$$

As a first step, use non-uniform fast fourier transform code to transform the overlap densities  $\Omega_{\mu \nu} (\mathbf{r})$ to $k$-space. To this end, we can also define a spherical grid in $k$-space using the qc-grid package.  